# 2-link arm RL visualization

상태는

$$
x=(q_1,q_2,\dot q_1,\dot q_2)\in\mathbb{R}^4
$$

이고 제어입력은 두 관절 torque

$$
u=(\tau_1,\tau_2)\in\mathbb{R}^2
$$

입니다.

동역학은

$$
M(q)\ddot q+C(q,\dot q)\dot q+g(q)+B\dot q=\tau
$$

를 사용합니다.

PyBullet은 Colab에서 DIRECT mode renderer로 사용하고, 같은 rollout을 $(q_1,q_2)$, $(q_1,\dot q_1)$, $(q_2,\dot q_2)$ 상태공간과 동기화해 MP4로 표시합니다.

세부 식과 코드 구성은 `docs/two_link_arm.md`를 참고합니다.

## SAC

replay buffer를 사용하는 off-policy actor-critic이며 entropy objective로 연속 torque action을 탐색합니다. early / middle / final snapshot을 따로 실행합니다.

In [ ]:
!pip -q install pybullet gymnasium stable-baselines3 imageio imageio-ffmpeg
!git clone -q https://github.com/HisameOgasahara/RL_tmp.git
%cd RL_tmp

import sys
sys.path.insert(0, ".")

from IPython.display import Video, display
import matplotlib.pyplot as plt


In [ ]:
from src.rl_tmp.two_link_arm.env import TwoLinkArmEnv

env_kwargs = {
    "dt": 0.04,
    "horizon": 150,
    "target": (1.25, 1.0),
    "initial_state": (0.0, 0.0, 0.0, 0.0),
    "random_reset": True,
}


In [ ]:
from src.rl_tmp.two_link_arm.sac import train_sac

model, snapshots = train_sac(
    env_kwargs=env_kwargs,
    total_timesteps=80_000,
    seed=0,
    device="auto",
)


In [ ]:
from src.rl_tmp.two_link_arm.render import (
    rollout_policy,
    render_rollout_video,
    stable_baselines_policy,
)

for name in ["early", "middle", "final"]:
    model.policy.load_state_dict(snapshots[name])
    env = TwoLinkArmEnv(**env_kwargs)

    policy = stable_baselines_policy(
        model,
        deterministic=False,
    )

    rollout = rollout_policy(
        policy,
        env,
        seed=123,
    )

    path = render_rollout_video(
        rollout,
        f"/content/two_link_sac_{name}.mp4",
        config=env.config,
        fps=25,
        frame_stride=2,
    )

    print(name)
    display(Video(str(path), embed=True))
